# 02 — Spraying Stage Explorer

Spray solution (acetone + ethylcellulose) is atomised onto the heated particle bed.
Acetone evaporates; coating polymer deposits on particle surfaces.

**Move the sliders** to explore how spray conditions affect:
- Product temperature (heat balance between inlet air and solvent evaporation)
- Acetone content on particles (accumulation vs. drying rate)
- Cumulative coating weight gain (%)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

from fluid_bed.parameters import ProcessParameters
from fluid_bed.models.spraying import run_spraying

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})


In [ ]:
FIXED = dict(
    particle_density = 1000.0,
    cp_particle      = 1400.0,
    diameter_bed     = 0.60,
    rho_air          = 1.10,
    cp_air           = 1010.0,
)
print("Fixed properties loaded.")


In [ ]:
S = dict(continuous_update=False)


def _run02(T_inlet_C=70.0, flow_m3h=300.0, spray_g_min=15.0, dmc_pct=15.0,
           atom_m3h=20.0, T_init_C=55.0, duration_min=30.0, batch_kg=2.0):
    rho   = FIXED["rho_air"]
    m_air = (flow_m3h / 3600.0) * rho
    m_atm = (atom_m3h / 3600.0) * rho
    T_inlet_K = T_inlet_C + 273.15
    T_init_K  = T_init_C  + 273.15

    params = ProcessParameters(
        **FIXED,
        diameter_eq          = 200e-6,
        batch_size           = batch_kg,
        air_flow_rates       = (m_air, m_air, m_air),
        air_temperatures     = (T_inlet_K, T_inlet_K, T_inlet_K),
        spray_rate           = spray_g_min / 60.0 / 1000.0,   # kg/s
        dry_matter_conc      = dmc_pct / 100.0,
        atomization_air_flow = m_atm,
    )

    res = run_spraying(params, duration=duration_min * 60.0, T_particle_init=T_init_K)
    t_min      = res.t / 60.0
    T_p_C      = res.T_particle - 273.15
    T_g_C      = res.T_gas      - 273.15
    acetone_pct = res.Y_particle * 100.0          # kg/kg → %
    wg_pct     = res.M_coating / batch_kg * 100.0 # weight gain %

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    ax = axes[0]
    ax.plot(t_min, T_g_C, "r--", lw=1.5, label="Gas T_g")
    ax.plot(t_min, T_p_C, "b-",  lw=2.5, label="Product T_p")
    ax.set_xlabel("Time (min)"); ax.set_ylabel("Temperature (°C)")
    ax.set_title("Temperatures"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(t_min, acetone_pct, color="darkorange", lw=2.5)
    ax.set_xlabel("Time (min)"); ax.set_ylabel("Acetone on particles (wt %)")
    ax.set_title("Particle acetone content"); ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(t_min, wg_pct, color="mediumpurple", lw=2.5)
    ax.set_xlabel("Time (min)"); ax.set_ylabel("Weight gain (%)")
    ax.set_title("Cumulative coating deposition"); ax.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()
    print(f"Final: T_product = {T_p_C[-1]:.1f} °C  |  "
          f"Acetone = {acetone_pct[-1]:.3f} %  |  "
          f"Weight gain = {wg_pct[-1]:.2f} %")


interact(
    _run02,
    T_inlet_C    = FloatSlider(70.0, min=40,  max=100, step=5,   description="T inlet (°C)",    **S),
    flow_m3h     = FloatSlider(300,  min=100, max=600, step=25,  description="Air flow (m³/h)", **S),
    spray_g_min  = FloatSlider(15.0, min=5,   max=50,  step=1,   description="Spray (g/min)",   **S),
    dmc_pct      = FloatSlider(15.0, min=5,   max=30,  step=1,   description="Dry matter (%)",  **S),
    atom_m3h     = FloatSlider(20.0, min=0,   max=80,  step=5,   description="Atom. air (m³/h)",**S),
    T_init_C     = FloatSlider(55.0, min=20,  max=80,  step=5,   description="T₀ product (°C)", **S),
    duration_min = FloatSlider(30.0, min=5,   max=90,  step=5,   description="Duration (min)",  **S),
    batch_kg     = FloatSlider(2.0,  min=0.5, max=5.0, step=0.5, description="Batch (kg)",      **S),
)
